# TERA — Combined LT + CALVIN Qualitative Figure

Generates a single composite PNG with:
- **Row 1–2**: Language-Table demonstrations
- **Row 3–4**: CALVIN demonstrations

Each row: **Instruction | Start → Middle → End** frames + **E_act | E_emb** tokens side-by-side.

**Run order:** Step 0 → Step 1 → Step 2 (download CALVIN once) → ⚙️ CONFIG → Steps 3–8.

## Step 0 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## Step 1 — Clone / update repo

In [ ]:
import os, sys
from pathlib import Path

REPO_DIR = '/content/RLConditionedVLA'

if Path(REPO_DIR).exists():
    print('Repo already cloned — pulling latest …')
    os.system(f'git -C {REPO_DIR} pull --ff-only')
else:
    print('Cloning repo …')
    os.system('git clone https://github.com/sara-kaz/RLConditionedVLA.git ' + REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Repo ready:', REPO_DIR)

## Step 2 — Download CALVIN dataset *(run once; skips if already on Drive)*

Downloads the **CALVIN task_D_D** split (~1 GB zip) from the official server and saves it to your Google Drive so you only need to do this once.

After extraction the layout will be:
```
MyDrive/VERA_CALVIN/task_D_D/
  training/
    episode_0000000.npz  …
    lang_annotations/auto_lang_ann.npy
  validation/
    …
```

> **If you already have CALVIN on Drive**, just set `CALVIN_DRIVE_DIR` below to the correct path and the cell will detect it and skip the download.

In [ ]:
import os, zipfile, shutil
from pathlib import Path

# ── Where to store CALVIN on your Drive (edit if needed) ──────────────────────
CALVIN_DRIVE_DIR = '/content/drive/MyDrive/VERA_CALVIN/task_D_D'

# Official CALVIN task_D_D download URL
CALVIN_URL = 'http://calvin.cs.uni-freiburg.de/dataset/task_D_D.zip'

# ── Check if already extracted on Drive ───────────────────────────────────────
training_dir = Path(CALVIN_DRIVE_DIR) / 'training'
ann_file     = training_dir / 'lang_annotations' / 'auto_lang_ann.npy'

if ann_file.exists() and len(list(training_dir.glob('episode_*.npz'))) > 10:
    n_eps = len(list(training_dir.glob('episode_*.npz')))
    print(f'✓ CALVIN already extracted at {CALVIN_DRIVE_DIR}')
    print(f'  {n_eps} .npz files found — skipping download.')
else:
    # ─────────────────────────────────────────────────────────────────────────
    # Download to LOCAL Colab disk first (fast — Colab's own internet ~100 MB/s)
    # then copy to Drive. Writing directly to Drive is 100x slower (~1 MB/s).
    # ─────────────────────────────────────────────────────────────────────────
    LOCAL_ZIP     = '/content/task_D_D.zip'
    LOCAL_EXTRACT = '/content/task_D_D'

    if not Path(LOCAL_ZIP).exists():
        print('Downloading CALVIN task_D_D (~1 GB) to local Colab disk …')
        print(f'  URL: {CALVIN_URL}')
        print('  Expected time: ~2–5 min (not 54 min — Drive is bypassed)')
        ret = os.system(f'wget -q --show-progress -O "{LOCAL_ZIP}" "{CALVIN_URL}"')
        if ret != 0 or not Path(LOCAL_ZIP).exists():
            raise RuntimeError('wget failed — check URL or internet connection')
        size_mb = Path(LOCAL_ZIP).stat().st_size / 1e6
        print(f'Download complete: {size_mb:.0f} MB')
    else:
        size_mb = Path(LOCAL_ZIP).stat().st_size / 1e6
        print(f'Zip already on local disk ({size_mb:.0f} MB) — skipping download.')

    # Extract locally (fast SSD — no Drive API overhead)
    if not Path(LOCAL_EXTRACT).exists():
        print('Extracting zip …')
        with zipfile.ZipFile(LOCAL_ZIP, 'r') as zf:
            members = zf.namelist()
            print(f'  {len(members):,} files in zip')
            for i, member in enumerate(members):
                zf.extract(member, '/content/')
                if i % 5000 == 0:
                    print(f'  extracted {i:,}/{len(members):,} …', end='\r')
        print(f'\nExtraction complete → {LOCAL_EXTRACT}')
    else:
        print(f'Already extracted locally at {LOCAL_EXTRACT}')

    # Copy to Drive for future sessions (one-time; Drive access is fast for reads)
    drive_parent = Path(CALVIN_DRIVE_DIR).parent
    drive_parent.mkdir(parents=True, exist_ok=True)
    if not Path(CALVIN_DRIVE_DIR).exists() and Path(LOCAL_EXTRACT).exists():
        print('Copying to Drive for future sessions (may take ~5 min) …')
        shutil.copytree(str(LOCAL_EXTRACT), CALVIN_DRIVE_DIR)
        n_eps = len(list(Path(CALVIN_DRIVE_DIR, 'training').glob('episode_*.npz')))
        print(f'✓ Copied to Drive — {n_eps} training episodes')

    # This session: point at the fast local copy
    CALVIN_DRIVE_DIR = LOCAL_EXTRACT if Path(LOCAL_EXTRACT).exists() else CALVIN_DRIVE_DIR
    n_eps = len(list(Path(CALVIN_DRIVE_DIR, 'training').glob('episode_*.npz')))
    print(f'✓ CALVIN ready at {CALVIN_DRIVE_DIR} — {n_eps} training episodes')

print('\nProceed to ⚙️ CONFIG.')

## ⚙️ CONFIG — Edit paths here before running

In [ ]:
# ── Language-Table ─────────────────────────────────────────────────────────────
LT_CHECKPOINT = '/content/drive/MyDrive/VERA_LT_Real/checkpoints/lt_full_vera/seed123/best_sft_vera.pt'
LT_CONFIG     = '/content/RLConditionedVLA/configs/config.yaml'
LT_DATA_ROOT  = '/content/lt_real/train'   # folder with episode_*/steps.pkl
LT_N_ROWS     = 2

# ── CALVIN (path set automatically by Step 2; change only if you moved the data) ──
CAL_CHECKPOINT = '/content/drive/MyDrive/VERA_CALVIN/checkpoints/calvin_vera/seed42/best_sft_vera.pt'
CAL_CONFIG     = '/content/RLConditionedVLA/configs/calvin_config.yaml'
CAL_DATA_ROOT  = '/content/drive/MyDrive/VERA_CALVIN/task_D_D'
CAL_SPLIT      = 'training'
CAL_N_ROWS     = 2

# ── Output ────────────────────────────────────────────────────────────────────
OUT_PNG = '/content/drive/MyDrive/corl_2026_template_submission/combined_lt_calvin_composite.png'
FIG_DPI = 240
SEED    = 42

# ── Figure typography ─────────────────────────────────────────────────────────
FIG_FONT_HDR   = 11.5
FIG_FONT_INSTR = 10.5
FIG_FONT_TOK   = 9.5
FIG_FONT_SECT  = 9.0
FIG_TOK_WRAP   = 38
FIG_INSTR_WRAP = 26
FIG_FRAME_PX   = 224
FIG_TOK_LINESPACING = 1.35

print('Config set.')

## Step 3 — Install dependencies

In [ ]:
import subprocess, sys

def pip_one(pkg, optional=False):
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                              stderr=subprocess.STDOUT)
        print(f'  ✓ {pkg[:70]}')
    except subprocess.CalledProcessError:
        if optional: print(f'  ⊘ optional: {pkg[:70]}')
        else: raise

for pkg in ('pyyaml', 'imageio', 'pillow', 'matplotlib', 'numpy', 'scipy'):
    pip_one(pkg)

try:
    import clip
    print('  ✓ clip (already installed)')
except ImportError:
    pip_one('git+https://github.com/openai/CLIP.git')

print('All dependencies ready.')

## Step 4 — Imports & shared utilities

In [ ]:
import random, textwrap, gc, pickle
from pathlib import Path

import numpy as np
import torch
import yaml
import clip
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image as PILImage
import torchvision.transforms as Tv

from models.vera_model import VERAModel

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cpu':
    print('  ⚠️  No GPU detected — Runtime → Change runtime type → T4 GPU')


def _load_cfg(ckpt_path, config_path):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if isinstance(ckpt, dict) and 'cfg' in ckpt:
        cfg = ckpt['cfg']; print('  config embedded in checkpoint')
    else:
        with open(config_path) as f: cfg = yaml.safe_load(f)
        print(f'  config from {config_path}')
    return ckpt, cfg


def _build_model(ckpt, cfg):
    m, v = cfg['model'], cfg.get('vera', {})
    model = VERAModel(
        num_actions=m['num_actions'], history_len=m['history_len'],
        num_vis_frames=m.get('num_vis_frames', 3),
        fusion_layers=m.get('fusion_layers', 6), fusion_heads=m.get('fusion_heads', 8),
        d_model=m.get('d_model', 256), d_ff_scale=m.get('d_ff_scale', 4),
        dropout=0.0, vision_token_dropout=0.0,
        freeze_clip=m.get('freeze_clip', True),
        unfreeze_clip_vision=m.get('unfreeze_clip_vision', False),
        use_lang_feedback=v.get('use_lang_feedback', True),
        use_temporal_history=v.get('use_temporal_history', True),
        use_reward_gate=v.get('use_reward_gate', True),
        use_consequence_token=v.get('use_consequence_token', True),
        action_dim=m.get('action_dim', 2),
        action_vocab=v.get('action_vocab'),
        chunk_size=m.get('chunk_size', 1),
    ).to(device)
    state = ckpt.get('model_state', ckpt) if isinstance(ckpt, dict) else ckpt
    missing, unexp = model.load_state_dict(state, strict=False)
    if missing: print(f'  missing keys: {len(missing)}')
    if unexp:   print(f'  unexpected keys: {len(unexp)}')
    model.eval()
    return model


def _make_transform(img_size=224):
    return Tv.Compose([
        Tv.Resize((img_size, img_size)), Tv.ToTensor(),
        Tv.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                     std =[0.26862954, 0.26130258, 0.27577711]),
    ])

def _f2t(frame, tf): return tf(PILImage.fromarray(frame.astype(np.uint8)))

def _thumb(frame):
    if frame is None: return np.full((FIG_FRAME_PX, FIG_FRAME_PX, 3), 210, dtype=np.uint8)
    img = PILImage.fromarray(np.asarray(frame, dtype=np.uint8))
    if img.size != (FIG_FRAME_PX, FIG_FRAME_PX):
        img = img.resize((FIG_FRAME_PX, FIG_FRAME_PX), PILImage.Resampling.LANCZOS)
    return np.asarray(img)

def _soft_wrap(text, width):
    text = (text or '').strip()
    return text if len(text) <= width else textwrap.fill(
        text, width=width, break_long_words=False, break_on_hyphens=False)

print('Imports OK.')

## Step 5 — Load LT model and collect 2 episodes

In [ ]:
print('Loading LT checkpoint …')
lt_ckpt, lt_cfg = _load_cfg(LT_CHECKPOINT, LT_CONFIG)
lt_model        = _build_model(lt_ckpt, lt_cfg)
lt_tf           = _make_transform(lt_cfg.get('data', {}).get('img_size', 224))

_lm = lt_cfg['model']; _lv = lt_cfg.get('vera', {})
LT_NUM_ACT = _lm['num_actions'];  LT_HIST_LEN = _lm['history_len']
LT_NUM_VIS = _lm.get('num_vis_frames', 3);  LT_ACT_DIM = _lm.get('action_dim', 2)
LT_NULL    = np.zeros(LT_ACT_DIM, dtype=np.float32)

LT_VOCAB = {0:'I pushed the object to the right',1:'I pushed the object up and to the right',
            2:'I pushed the object upward',3:'I pushed the object up and to the left',
            4:'I pushed the object to the left',5:'I pushed the object down and to the left',
            6:'I pushed the object downward',7:'I pushed the object down and to the right'}
if _lv.get('action_vocab'): LT_VOCAB = {int(k): v for k, v in _lv['action_vocab'].items()}

def lt_verbalize(r, d):
    if r > 0.8: return 'I moved significantly closer to the goal and received a high reward.'
    if r > 0.3: return 'I moved closer to the goal and received a moderate reward.'
    if d < -0.005: return 'I moved closer to the goal but received a low reward.'
    if d >  0.005: return 'I moved away from the goal and received a low reward.'
    return 'I made little progress and received a low reward.'

def _lt_disc(av):
    dx, dy = float(av[0]), float(av[1])
    if abs(dx) < 1e-3 and abs(dy) < 1e-3: return LT_NUM_ACT
    return int(round(np.arctan2(dy, dx) / (np.pi / 4))) % LT_NUM_ACT

def _lt_frame(step):
    for k in ('obs','image','rgb','pixels','frame'):
        v = step.get(k)
        if v is None: continue
        if isinstance(v, dict):
            for sk in ('rgb','image','pixels'):
                v2 = v.get(sk)
                if v2 is not None and isinstance(v2,np.ndarray) and v2.ndim==3: return v2.astype(np.uint8)
        elif isinstance(v,np.ndarray) and v.ndim==3: return v.astype(np.uint8)
    return None

_lt_tc = {}
def _lt_tokens(steps, t, instr):
    if instr not in _lt_tc: _lt_tc[instr] = clip.tokenize([instr])[0]
    lang_in = _lt_tc[instr].unsqueeze(0).to(device)
    ah, rh, avh = [], [], []
    for j in range(max(0,t-LT_HIST_LEN),t):
        av=np.asarray(steps[j].get('action',steps[j].get('action_vec',[0,0])),dtype=np.float32).flatten()[:2]
        if np.abs(av).max()>1.0: av=av/np.abs(av).max()
        ah.append(_lt_disc(av)); rh.append(float(steps[j].get('reward',0.0))); avh.append(np.clip(av,-1,1))
    while len(ah)<LT_HIST_LEN: ah.insert(0,LT_NUM_ACT); rh.insert(0,0.0); avh.insert(0,LT_NULL.copy())
    prev_a=ah[-1] if t>0 else LT_NUM_ACT; prev_r=rh[-1] if t>0 else 0.0
    prev_d=float(steps[t-1].get('state_delta',0.0) or 0.0) if t>0 else 0.0
    fts=[]
    for fi in [max(0,t-2),max(0,t-1),t]:
        fr=_lt_frame(steps[fi])
        if fr is not None: fts.append(_f2t(fr,lt_tf))
    if not fts: return '—','—'
    while len(fts)<LT_NUM_VIS: fts.insert(0,torch.zeros_like(fts[0]))
    with torch.no_grad():
        out=lt_model(torch.stack(fts[-LT_NUM_VIS:]).unsqueeze(0).to(device),lang_in,
                     torch.tensor([ah],dtype=torch.long).to(device),
                     torch.tensor([rh],dtype=torch.float32).to(device),
                     torch.tensor([prev_a],dtype=torch.long).to(device),
                     torch.tensor([prev_r],dtype=torch.float32).to(device),
                     state_delta=torch.tensor([prev_d],dtype=torch.float32).to(device),
                     action_vec_hist=torch.tensor([np.stack(avh)],dtype=torch.float32).to(device))
        action=int(out['logits'].argmax(dim=-1).item())
    return LT_VOCAB.get(action,f'I performed action {action}'), lt_verbalize(float(steps[t].get('reward',0.0)),prev_d)

def collect_lt(root, n):
    root=Path(root)
    if not root.is_dir(): print(f'[LT] Not found: {root}'); return []
    meta=[]
    for ep_dir in sorted(root.glob('episode_*')):
        pkl=ep_dir/'steps.pkl'
        if not pkl.is_file(): continue
        with open(pkl,'rb') as f: steps=pickle.load(f)
        if len(steps)<4: continue
        instr=''
        for k in ('instruction','language_instruction','task'):
            v=steps[0].get(k)
            if v: instr=(v.decode() if isinstance(v,bytes) else str(v)).strip().rstrip('.'); break
        total_r=sum(float(s.get('reward',0)) for s in steps)
        meta.append((total_r,instr,ep_dir)); del steps
    meta.sort(key=lambda x:-x[0])
    picked,used=[],set()
    for total_r,instr,ep_dir in meta:
        key=instr.lower()[:40]
        if key in used: continue
        used.add(key)
        with open(ep_dir/'steps.pkl','rb') as f: steps=pickle.load(f)
        T=len(steps); mid=T//2
        frames=[_lt_frame(steps[i]) for i in [0,mid,T-1]]
        if any(f is None for f in frames): continue
        nar,know=_lt_tokens(steps,mid,instr)
        del steps; gc.collect()
        picked.append({'dataset':'Language-Table','instruction':instr,
                       'start':frames[0],'mid':frames[1],'end':frames[2],
                       'narration_mid':nar,'knowledge_mid':know})
        print(f'  [LT {len(picked)}] "{instr}"  reward={total_r:.2f}')
        if len(picked)>=n: break
    return picked

print('\nCollecting LT episodes …')
lt_episodes = collect_lt(LT_DATA_ROOT, LT_N_ROWS)
print(f'Got {len(lt_episodes)}/{LT_N_ROWS} LT episodes.')

## Step 6 — Load CALVIN model and collect 2 episodes

In [ ]:
print('Loading CALVIN checkpoint …')
cal_ckpt, cal_cfg = _load_cfg(CAL_CHECKPOINT, CAL_CONFIG)
cal_model         = _build_model(cal_ckpt, cal_cfg)
cal_tf            = _make_transform(cal_cfg.get('data', {}).get('img_size', 224))

_cm=cal_cfg['model']; _cv=cal_cfg.get('vera',{})
CAL_NUM_ACT=_cm['num_actions']; CAL_HIST_LEN=_cm['history_len']
CAL_NUM_VIS=_cm.get('num_vis_frames',3); CAL_ACT_DIM=_cm.get('action_dim',7)
CAL_NULL=np.zeros(CAL_ACT_DIM,dtype=np.float32)

_DEFAULT_CAL_VOCAB={0:'I moved the end-effector to the right',1:'I moved the end-effector to the left',
    2:'I moved the end-effector forward',3:'I moved the end-effector backward',
    4:'I moved the end-effector upward',5:'I moved the end-effector downward',
    6:'I rotated the wrist clockwise',7:'I rotated the wrist counterclockwise',
    8:'I pitched the end-effector forward',9:'I pitched the end-effector backward',
    10:'I yawed the end-effector to the left',11:'I yawed the end-effector to the right',
    12:'I opened the gripper',13:'I closed the gripper'}
CAL_VOCAB={int(k):v for k,v in (_cv.get('action_vocab') or _DEFAULT_CAL_VOCAB).items()}

def cal_verbalize(done,mag):
    if done>=1.0: return 'I completed the sub-task and received a success signal.'
    if mag>0.3:   return 'I made a large movement but the task is not yet complete.'
    if mag>0.05:  return 'I made progress toward the goal but have not finished.'
    return 'I made a small adjustment with no task completion yet.'

def _cal_disc(ra):
    if ra[6]>0.5: return 12
    if ra[6]<-0.5: return 13
    dom=int(np.argmax(np.abs(ra[:6])))
    return dom*2+(0 if ra[dom]>=0 else 1)

_cal_tc={}
def _cal_tokens(frames,act_idx,rewards,avecs,t,instr):
    if instr not in _cal_tc: _cal_tc[instr]=clip.tokenize([instr])[0]
    lang_in=_cal_tc[instr].unsqueeze(0).to(device)
    ah,rh,avh=[],[],[]
    for j in range(max(0,t-CAL_HIST_LEN),t):
        ah.append(int(act_idx[j])); rh.append(float(rewards[j]))
        avh.append(np.clip(avecs[j][:CAL_ACT_DIM],-1,1))
    while len(ah)<CAL_HIST_LEN: ah.insert(0,CAL_NUM_ACT); rh.insert(0,0.0); avh.insert(0,CAL_NULL.copy())
    fts=[_f2t(frames[max(0,t-2)],cal_tf),_f2t(frames[max(0,t-1)],cal_tf),_f2t(frames[t],cal_tf)]
    while len(fts)<CAL_NUM_VIS: fts.insert(0,torch.zeros_like(fts[0]))
    with torch.no_grad():
        out=cal_model(torch.stack(fts[-CAL_NUM_VIS:]).unsqueeze(0).to(device),lang_in,
                      torch.tensor([ah],dtype=torch.long).to(device),
                      torch.tensor([rh],dtype=torch.float32).to(device),
                      torch.tensor([ah[-1]],dtype=torch.long).to(device),
                      torch.tensor([rh[-1]],dtype=torch.float32).to(device),
                      state_delta=torch.tensor([0.0],dtype=torch.float32).to(device),
                      action_vec_hist=torch.tensor([np.stack(avh)],dtype=torch.float32).to(device))
        action=int(out['logits'].argmax(dim=-1).item())
    return CAL_VOCAB.get(action,f'I performed action {action}'), cal_verbalize(float(rewards[t]),float(np.linalg.norm(avecs[t])))

def collect_calvin(root, split, n):
    root=Path(root)/split
    if not root.is_dir(): print(f'[CALVIN] Not found: {root}'); return []
    la_path=root/'lang_annotations'/'auto_lang_ann.npy'
    if not la_path.exists(): print(f'[CALVIN] No lang_annotations at {la_path}'); return []
    la=np.load(la_path,allow_pickle=True).item()
    indx=la['info']['indx']; tasks=la['language']['task']
    eps=sorted(root.glob('episode_*.npz'))
    avail={int(f.stem.split('_')[1]):f for f in eps}
    print(f'[CALVIN] {len(eps)} npz files, {len(indx)} annotated episodes')
    scored=[]
    for (s,e),task in zip(indx,tasks):
        indices=list(range(s,e+1))
        if not all(i in avail for i in indices): continue
        last_done=float(np.load(avail[indices[-1]],allow_pickle=True).get('done',0))
        scored.append((last_done,task,s,e))
    scored.sort(key=lambda x:-x[0])
    picked,used=[],set()
    for score,task,s,e in scored:
        key=task.lower()[:40]
        if key in used: continue
        used.add(key)
        indices=list(range(s,e+1))
        frames,aidx,rews,avecs=[],[],[],[]
        ok=True
        for idx in indices:
            try: data=np.load(avail[idx],allow_pickle=True)
            except: ok=False; break
            fr=data.get('rgb_static')
            if fr is None: ok=False; break
            ra=np.asarray(data.get('rel_actions',np.zeros(7)),dtype=np.float32).flatten()[:7]
            frames.append(np.asarray(fr,dtype=np.uint8)); aidx.append(_cal_disc(ra))
            rews.append(float(data.get('done',0))); avecs.append(ra)
        if not ok or len(frames)<3: continue
        T=len(frames); mid=T//2
        nar,know=_cal_tokens(frames,np.array(aidx),np.array(rews),np.stack(avecs),mid,task)
        gc.collect()
        picked.append({'dataset':'CALVIN','instruction':task,
                       'start':frames[0],'mid':frames[mid],'end':frames[-1],
                       'narration_mid':nar,'knowledge_mid':know})
        print(f'  [CAL {len(picked)}] "{task}"  done={score:.0f}')
        if len(picked)>=n: break
    return picked

print('\nCollecting CALVIN episodes …')
cal_episodes = collect_calvin(CAL_DATA_ROOT, CAL_SPLIT, CAL_N_ROWS)
print(f'Got {len(cal_episodes)}/{CAL_N_ROWS} CALVIN episodes.')

## Step 7 — Build and save the combined figure

In [ ]:
all_episodes = lt_episodes + cal_episodes
N_ROWS = len(all_episodes)
assert N_ROWS > 0, 'No episodes collected — check dataset paths in CONFIG cell.'

FIG_W=11.0; HDR_H,IMG_H,TOK_H=0.22,2.20,0.52
total_h=HDR_H+N_ROWS*(IMG_H+TOK_H)+0.08

plt.rcParams.update({'font.family':'sans-serif','font.size':FIG_FONT_INSTR,
                     'axes.titlesize':FIG_FONT_HDR,'figure.dpi':FIG_DPI})

LT_BG=['#EEF3FA','#F7FAFF']; CAL_BG=['#F5F2EE','#FBF8F5']
TEXT_CLR='#1a2744'; TOKEN_CLR='#1a2744'; COL_TITLES=['Start','Middle','End']

height_ratios=[HDR_H]+[IMG_H,TOK_H]*N_ROWS
fig=plt.figure(figsize=(FIG_W,total_h),facecolor='white',dpi=FIG_DPI)
master=gridspec.GridSpec(1+N_ROWS*2,4,figure=fig,
    width_ratios=[1.05,1,1,1],height_ratios=height_ratios,
    left=0.06,right=0.99,top=0.96,bottom=0.04,wspace=0.03,hspace=0.04)

def _draw(ax,frame,bg):
    ax.set_facecolor(bg); ax.imshow(_thumb(frame)); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_linewidth(0.5); sp.set_edgecolor('#CCCCCC')

ax_h0=fig.add_subplot(master[0,0]); ax_h0.axis('off')
ax_h0.text(0.98,0.5,'Instruction',ha='right',va='center',
           fontsize=FIG_FONT_HDR,fontweight='bold',color=TEXT_CLR)
for j,t in enumerate(COL_TITLES):
    ax_h=fig.add_subplot(master[0,j+1]); ax_h.axis('off')
    ax_h.text(0.5,0.5,t,ha='center',va='center',
              fontsize=FIG_FONT_HDR,fontweight='bold',color=TEXT_CLR)

lt_n=len(lt_episodes)
for row_i,ep in enumerate(all_episodes):
    r_img=1+row_i*2; r_tok=r_img+1
    is_cal=ep['dataset']=='CALVIN'
    local_i=row_i if not is_cal else row_i-lt_n
    bg=(CAL_BG if is_cal else LT_BG)[local_i%2]
    ax_lbl=fig.add_subplot(master[r_img:r_tok+1,0])
    ax_lbl.set_facecolor(bg); ax_lbl.axis('off')
    if row_i==0 or (is_cal and row_i==lt_n):
        ax_lbl.text(0.98,0.95,'CALVIN' if is_cal else 'Language-Table',
                    transform=ax_lbl.transAxes,fontsize=FIG_FONT_SECT,
                    ha='right',va='top',color='#888888',style='italic',clip_on=False)
    instr=ep['instruction'].strip().rstrip('.')
    if instr: instr=instr[0].upper()+instr[1:]
    ax_lbl.text(0.96,0.5,_soft_wrap(instr+'.',FIG_INSTR_WRAP),
                transform=ax_lbl.transAxes,fontsize=FIG_FONT_INSTR,
                ha='right',va='center',color=TEXT_CLR,linespacing=1.25,clip_on=False)
    for col,key in [(1,'start'),(2,'mid'),(3,'end')]:
        _draw(fig.add_subplot(master[r_img,col]),ep[key],bg)
    ax_tok=fig.add_subplot(master[r_tok,1:4])
    ax_tok.set_facecolor(bg); ax_tok.axis('off')
    ax_tok.set_ylim(0,1); ax_tok.set_xlim(0,1); ax_tok.margins(x=0.02,y=0.10)
    ax_tok.text(0.01,0.5,
                f"$E_{{\\mathrm{{act}}}}$: {_soft_wrap(ep.get('narration_mid','—'),FIG_TOK_WRAP)}",
                transform=ax_tok.transAxes,fontsize=FIG_FONT_TOK,
                ha='left',va='center',color=TOKEN_CLR,linespacing=FIG_TOK_LINESPACING,clip_on=False)
    ax_tok.axvline(0.50,color='#cccccc',linewidth=0.8,clip_on=False)
    ax_tok.text(0.52,0.5,
                f"$E_{{\\mathrm{{emb}}}}$: {_soft_wrap(ep.get('knowledge_mid','—'),FIG_TOK_WRAP)}",
                transform=ax_tok.transAxes,fontsize=FIG_FONT_TOK,
                ha='left',va='center',color=TOKEN_CLR,linespacing=FIG_TOK_LINESPACING,clip_on=False)
    if row_i==lt_n-1 and len(cal_episodes)>0:
        ax_tok.axhline(0.0,color='#aaaaaa',linewidth=1.2,xmin=0,xmax=1,clip_on=False,zorder=10)

Path(OUT_PNG).parent.mkdir(parents=True,exist_ok=True)
fig.savefig(OUT_PNG,dpi=FIG_DPI,bbox_inches='tight',facecolor='white')
print(f'Saved to Drive: {OUT_PNG}')
fig.savefig('/content/combined_lt_calvin_composite.png',dpi=FIG_DPI,bbox_inches='tight',facecolor='white')
plt.show()
print('Done!')

## Step 8 — Preview

In [ ]:
from IPython.display import Image as IPyImage
IPyImage('/content/combined_lt_calvin_composite.png')